[this doc on github](https://github.com/dotnet/interactive/tree/main/samples/notebooks/csharp/Docs)

# HorizonteNet


## Jupyter Notebooks

Un Jupyter Notebook con C# es un entorno interactivo que permite escribir y ejecutar código C# en bloques llamados celdas. Además de código, permite combinar explicaciones, fórmulas, gráficos y resultados, lo que facilita la experimentación, el análisis de datos y el aprendizaje de programación de forma interactiva.


En este ejemplo se muestran dos formas de proporcionar una salida en un Jupyter Notebook con C#:
* display(x);: muestra explícitamente el contenido de la variable x como salida de la celda.
* x: al dejar una expresión como última línea de la celda, Jupyter muestra automáticamente su valor.

In [ ]:
var x = "HolaMundo!";
display(x);
x

## Integración con el entorno modular

HorizonteNet está disponible en los cuadernos.

Seleccionando el Kernel 'Default' el proveedor de servicios esta disponible a través del objeto ServiceProvider

En este ejemplo obtenemos el servicio para gestionar los comandos y ejecutamos

In [ ]:
using Horizonte;
using Microsoft.Extensions.DependencyInjection;
var gescom = ServiceProvider.GetService<IHGesCom>();
var res = gescom.RunCommand("Workers_GetAvailableWorkers");
res

### Registro de nugets

Es posible referenciar paquetes nugets con:

#r "nuget:[nombre nuget],[versión]"

Y a dll en disco

#r "[ruta/a/archivo.dll]"


En este ejemplo referenciamos las abstracciones de logging, obtenemos el servicio de log y escribimos mensaje en el registro

In [ ]:
#r "nuget:microsoft.extensions.logging.abstractions,10.0.8"

using Microsoft.Extensions.Logging;
using Microsoft.Extensions.DependencyInjection;

var log = ServiceProvider.GetService<ILogger>();
log.LogInformation("***** jupyter log *****")

### Contexto del entorno modular

Es contexto del entorno es un diccionario <string,object> de lectura y escritura, persistente en el archivo horizonte.json.

Para el ejemplo, primero creamos la entidad para almacenar la configuración.

In [ ]:
using Horizonte;
using Horizonte.Interfaces;

public class TestConfig
{
    public string Name {get;set;} = "hola mundo";
    public int Value {get;set;} = 0;
    public string Secret {get;set;} = String.Empty;
}


Asignamos valores a la configuración y con la función Update del servicio IhContext establecemos la configuración en el contexto.


In [ ]:
TestConfig config = new TestConfig();
config.Name="jupyter value";
ServiceProvider.GetService<IhContext>().Update<TestConfig>(config);
config

La función Get de iHContext nos permite recuperar la sección de configuración del contexto.

In [ ]:
var readconfig = ServiceProvider.GetService<IhContext>().Get<TestConfig>();
readconfig.Name

### Credenciales

Las funciones SetCredential y GetCredential del servicio IHCredManager encriptan y desencriptan los valores de las credenciales

In [ ]:
ServiceProvider.GetService<IHCredManager>().SetCredential("testcred","clearpass");

var secret = ServiceProvider.GetService<IHCredManager>().GetCredential("testcred");

secret

### Manejo de los BackgrounServices

Podemos listar, iniciar y detener los servicios 

In [ ]:
var services = gescom.RunCommand("Workers_GetAvailableWorkers");
services

In [ ]:
var isrunning = gescom.RunCommand("Workers_IsRunning",["WebWorker"]);
isrunning


In [ ]:
gescom.RunCommand("Workers_StartService",["WebWorker"]);

In [ ]:
gescom.RunCommand("Workers_StopService",["WebWorker"]);

### Integración de Widgets

Es posible insertar Widgets en los cuadernos.

Los comandos con el rol 'widget' devuelven un objeto WidgetDef, el notebook es capaz de renderizar el objeto incluido en la definición WidgetDef.

In [ ]:
var commandwidget = gescom.RunCommand("HorizonteConfig_Commands");
commandwidget


Podemos crear por código la definición WidgetDef.

En este ejemplo, primero referenciamos los paqutes y espacios de nombre que necesitamos:

In [ ]:
#r "nuget:Horizonte.WorkFlows,10.0.0-beta"
#r "nuget:Horizonte.Extension.AiWorkFlows,10.0.0-beta"
using Horizonte.WorkFlows;
using Horizonte.WorkFlows.Widgets;
using Horizonte.Extension.AiWorkFlows;

En el ejemplo, vamos a mostrar un WorkFlow definido por código.

Primero creamos el WorkFlow, luego creamos la definición del widget que requiere del Tipo a representar, en este caso Horizonte.WorkFlows.Widgets.WorkFlowDesigner que representa el archivo WorkFlowDesigner.razor, y en el diccionario <string,object> de los parámetros incluimos los parámetros del componente razor

Finalmente devolvemos el WidgetDef para que sea representado en el cuaderno.

In [ ]:
//qutiamos advertencia Experimental
#pragma warning disable HORZEXP001

//workflow
var workflow = new WorkFlowDef();
workflow.Id = "jupyer";
workflow.Name ="JupyterWorkFlow";
workflow.MessageType ="System.String";
workflow.Nodes.Add(new WorkFlowNode(){
    Id ="nodein",
    Name="Nodo Inicio",
    Type=WorkFlowNodeTypes.In,
    PosX=150,
    PosY=150,
    CommandAction="comando_a_ejecutar"

});


// widget
var widgetcode = new WidgetDef()
{
    Type = typeof(Horizonte.WorkFlows.Widgets.WorkFlowDesigner)
};
widgetcode.Parameters = new Dictionary<string,object>();
widgetcode.Parameters.Add("WorkFlowDef",workflow);
widgetcode.Parameters.Add("OnSave",null);
widgetcode.Parameters.Add("OnDelete",null);
widgetcode.Parameters.Add("DesignerMode",DesignerMode.Design);

#pragma warning restore HORZEXP001

//mostrar
widgetcode




### Ejecución de comandos

Los comandos pueden ser asíncronos o no.

Podemos identificar sí el comando es asíncrono y ejecutar

In [ ]:
//var command = "MemoryLog_IsEnabled";
var command = "Sample_Tarea";

if(!gescom.ExistCommand(command)) {return "no existe";}

if(gescom.IsAsyncCommand(command))
{
    display("Es asíncrono");
    var result = gescom.RunCommandAsync(command).GetAwaiter().GetResult();
    return result;
}
else
{
    display("NO es asíncrono");
    var result = gescom.RunCommand(command);
    return result;

}



Es posible definir el tipo devuelto por el comando:

In [ ]:
using Horizonte.Entities;
var restipado = gescom.RunCommand<List<WorkerDefItem>>("Workers_GetAvailableWorkers");
restipado[0].WorkerType


Exite una versión equivalente a RunCommand para trabajar con Json, los parámetros y la respuesta son cadenas de texto con los objetos serializados en json.

In [ ]:
var resjson = gescom.RunCommandJson("Workers_GetAvailableWorkers",null);
resjson

### Roles
A cada comando se le puede asignar un rol, un rol es una agrupación de funciones con la misma firma de función y por lo tanto intercambiables.

Hay definidos algunos roles, por ejemplo el rol 'widget' agrupa funciones con la firma:
 public WidgetDef NombreComando(){}

Podemos listar los comandos intercambiables de un determinado rol con la función 'GetRoleCommands'

In [ ]:
var rolecom = gescom.GetRoleCommands("widget");
rolecom